<a href="https://colab.research.google.com/github/SakTauseef/SPARSE-AUTOENCODERS-FIND-HIGHLY-INTERPRETABLE-FEATURES-IN-LANGUAGE-MODELS/blob/main/sae_reproduction_starter_(2).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Sparse autoencoder reproduction — starter notebook

Based on Cunningham, Ewart, Riggs, Huben & Sharkey (2023), *"Sparse Autoencoders Find Highly Interpretable Features in Language Models"* (arXiv:2309.08600).

Official reference repo: https://github.com/HoagyC/sparse_coding — used here as a reference only; everything below is written from scratch. Log any code you copy or adapt from it in your own `PROVENANCE.md`.

**How to run:** Runtime -> Change runtime type -> select T4 GPU, then Run All. Each section matches a step in Stage 3 of the assignment brief.

## 0. Environment setup — pin versions first

In [1]:
!pip install -q transformers==4.36.2 datasets==2.16.1 accelerate==0.25.0

# write these to requirements.txt for your repo so the environment is reproducible
with open("requirements.txt", "w") as f:
    f.write("transformers==4.36.2\ndatasets==2.16.1\naccelerate==0.25.0\ntorch\n")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 126.8/126.8 kB 5.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.2/8.2 MB 102.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 507.1/507.1 kB 43.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 265.7/265.7 kB 30.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 115.3/115.3 kB 14.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 166.4/166.4 kB 19.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 46.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 99.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 135.4/135.4 kB 16.1 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
sentence-transformers 5.7.0 requires tokenizers>=0.19, but you have tokenizers 0.15.2 which is incompat

In [2]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from transformers import AutoTokenizer, AutoModelForCausalLM
from datasets import load_dataset

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Using device:", device)

/usr/local/lib/python3.13/dist-packages/transformers/utils/generic.py:441: FutureWarning: `torch.utils._pytree._register_pytree_node` is deprecated. Please use `torch.utils._pytree.register_pytree_node` instead.
  _torch_pytree._register_pytree_node(


Using device: cuda


## 1. Load the frozen model
Pythia-70M, weights frozen — we never train this model, only read its internal activations.

In [3]:
MODEL_NAME = "EleutherAI/pythia-70m"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(MODEL_NAME).to(device)
model.eval()
for p in model.parameters():
    p.requires_grad = False

print(model.config)

/usr/local/lib/python3.13/dist-packages/huggingface_hub/file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/396 [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/99.0 [00:00<?, ?B/s]

Special tokens have been added in the vocabulary, make sure the associated word embeddings are fine-tuned or trained.


config.json:   0%|          | 0.00/567 [00:00<?, ?B/s]

/usr/local/lib/python3.13/dist-packages/transformers/utils/generic.py:309: FutureWarning: `torch.utils._pytree._register_pytree_node` is deprecated. Please use `torch.utils._pytree.register_pytree_node` instead.
  _torch_pytree._register_pytree_node(
/usr/local/lib/python3.13/dist-packages/transformers/utils/generic.py:309: FutureWarning: `torch.utils._pytree._register_pytree_node` is deprecated. Please use `torch.utils._pytree.register_pytree_node` instead.
  _torch_pytree._register_pytree_node(
/usr/local/lib/python3.13/dist-packages/huggingface_hub/file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


model.safetensors:   0%|          | 0.00/166M [00:00<?, ?B/s]

GPTNeoXConfig {
  "_name_or_path": "EleutherAI/pythia-70m",
  "architectures": [
    "GPTNeoXForCausalLM"
  ],
  "attention_dropout": 0.0,
  "bos_token_id": 0,
  "classifier_dropout": 0.1,
  "eos_token_id": 0,
  "hidden_act": "gelu",
  "hidden_dropout": 0.0,
  "hidden_size": 512,
  "initializer_range": 0.02,
  "intermediate_size": 2048,
  "layer_norm_eps": 1e-05,
  "max_position_embeddings": 2048,
  "model_type": "gpt_neox",
  "num_attention_heads": 8,
  "num_hidden_layers": 6,
  "rope_scaling": null,
  "rotary_emb_base": 10000,
  "rotary_pct": 0.25,
  "tie_word_embeddings": false,
  "torch_dtype": "float16",
  "transformers_version": "4.36.2",
  "use_cache": true,
  "use_parallel_residual": true,
  "vocab_size": 50304
}



## 2. Collect real activations
We use a small slice of WikiText as a stand-in for the Pile (note this substitution in your report). `N_DOCS` is deliberately tiny for the first pass — scale it up once everything below works.

In [4]:
LAYER = 3        # which hidden_states index to probe (0 = embeddings, i = output of layer i)
N_DOCS = 200      # small for sanity check; try 5000+ once this notebook runs cleanly
MAX_LENGTH = 128

dataset = load_dataset("wikitext", "wikitext-103-raw-v1", split="train", streaming=True)

texts = []
for example in dataset:
    if len(example["text"].strip()) > 20:
        texts.append(example["text"])
    if len(texts) >= N_DOCS:
        break

print(f"Collected {len(texts)} text samples")
print(texts[0][:200])

Collected 200 text samples
 = Valkyria Chronicles III = 



In [5]:
@torch.no_grad()
def collect_activations(texts, layer, batch_size=8):
    all_acts = []
    for i in range(0, len(texts), batch_size):
        batch = texts[i:i + batch_size]
        inputs = tokenizer(
            batch, return_tensors="pt", padding=True,
            truncation=True, max_length=MAX_LENGTH
        ).to(device)
        outputs = model(**inputs, output_hidden_states=True)
        hidden = outputs.hidden_states[layer]        # (batch, seq_len, d_model)
        mask = inputs["attention_mask"].bool()
        acts = hidden[mask]                            # drop padding, flatten to (n_tokens, d_model)
        all_acts.append(acts.cpu())
    return torch.cat(all_acts, dim=0)

activations = collect_activations(texts, LAYER)
print("Activation tensor shape:", activations.shape)
torch.save(activations, f"activations_layer{LAYER}.pt")

Activation tensor shape: torch.Size([11442, 512])


## 3. The sparse autoencoder
Directly implements the paper's equations:

`c = ReLU(Mx + b)`  ·  `x_hat = M^T c`  ·  `loss = ||x - x_hat||^2 + alpha * ||c||_1`

In [6]:
class SparseAutoencoder(nn.Module):
    def __init__(self, d_model, dict_size):
        super().__init__()
        self.W = nn.Parameter(torch.randn(dict_size, d_model) * 0.1)
        self.b_enc = nn.Parameter(torch.zeros(dict_size))
        with torch.no_grad():
            self.W.data = F.normalize(self.W.data, dim=1)  # row-normalize (paper Appendix B)

    def encode(self, x):
        return F.relu(x @ self.W.T + self.b_enc)

    def decode(self, c):
        return c @ self.W

    def forward(self, x):
        c = self.encode(x)
        x_hat = self.decode(c)
        return x_hat, c


def sae_loss(x, x_hat, c, alpha):
    recon_loss = ((x - x_hat) ** 2).sum(dim=-1).mean()
    sparsity_loss = c.abs().sum(dim=-1).mean()
    return recon_loss + alpha * sparsity_loss, recon_loss, sparsity_loss

In [7]:
@torch.no_grad()
def top_activating_texts(sae, texts, feature_idx, layer=LAYER, top_k=5):
    results = []
    for text in texts:
        inputs = tokenizer(text, return_tensors="pt", truncation=True, max_length=MAX_LENGTH).to(device)
        hidden = model(**inputs, output_hidden_states=True).hidden_states[layer][0]
        c = sae.encode(hidden)
        max_act = c[:, feature_idx].max().item()
        results.append((max_act, text))
    results.sort(reverse=True, key=lambda r: r[0])
    return results[:top_k]

## 4. Sanity check — one batch, check shapes, check loss is finite
Do not skip this. Fix anything wrong here before training anything for real.

In [8]:
d_model = activations.shape[-1]
R = 4                     # dictionary size ratio (paper sweeps R in {0.5, 1, 2, 4, 8})
dict_size = d_model * R
alpha = 8e-4               # sparsity strength (paper uses values like 8.6e-4)

sae = SparseAutoencoder(d_model, dict_size).to(device)

batch = activations[:64].to(device)
x_hat, c = sae(batch)
loss, recon, sparsity = sae_loss(batch, x_hat, c, alpha)

print("input shape:  ", batch.shape)
print("x_hat shape:  ", x_hat.shape)
print("c shape:      ", c.shape)
print("loss:         ", loss.item(), "| finite:", torch.isfinite(loss).item())

input shape:   torch.Size([64, 512])
x_hat shape:   torch.Size([64, 512])
c shape:       torch.Size([64, 2048])
loss:          3106.291748046875 | finite: True


## 5. Tiny-subset training — confirm the loss actually goes down

In [9]:
def train_sae(sae, data, alpha, epochs=30, batch_size=64, log_every=5):
    optimizer = torch.optim.Adam(sae.parameters(), lr=1e-3)
    n = data.shape[0]
    history = []
    for epoch in range(epochs):
        perm = torch.randperm(n)
        total_loss = total_recon = total_sparsity = 0.0
        for i in range(0, n, batch_size):
            idx = perm[i:i + batch_size]
            x = data[idx].to(device)
            x_hat, c = sae(x)
            loss, recon, sparsity = sae_loss(x, x_hat, c, alpha)

            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
            with torch.no_grad():
                sae.W.data = F.normalize(sae.W.data, dim=1)

            total_loss += loss.item()
            total_recon += recon.item()
            total_sparsity += sparsity.item()

        history.append((epoch, total_loss, total_recon, total_sparsity))
        if epoch % log_every == 0 or epoch == epochs - 1:
            print(f"epoch {epoch:3d} | loss {total_loss:8.3f} | recon {total_recon:8.3f} | sparsity {total_sparsity:8.3f}")
    return history


tiny_subset = activations[:500]
sae = SparseAutoencoder(d_model, dict_size).to(device)
history = train_sae(sae, tiny_subset, alpha, epochs=30)

epoch   0 | loss 3366.734 | recon 3364.007 | sparsity 3408.423
epoch   5 | loss  299.735 | recon  298.112 | sparsity 2028.399
epoch  10 | loss  163.418 | recon  161.766 | sparsity 2064.485
epoch  15 | loss  118.725 | recon  117.092 | sparsity 2041.088
epoch  20 | loss   85.968 | recon   84.333 | sparsity 2043.351
epoch  25 | loss   99.778 | recon   98.150 | sparsity 2035.128
epoch  29 | loss   80.182 | recon   78.557 | sparsity 2032.049


## 5b. Diagnostic — how many drawers are actually open?
`recon` improving while `sparsity` plateaus (as you likely just saw) means the model is reconstructing well but not specializing features. This checks the actual number of active features per token — a genuinely sparse result should be roughly 10-50 out of `dict_size`, not hundreds.

In [10]:
@torch.no_grad()
def active_features_per_token(sae, data, sample_size=200):
    x = data[:sample_size].to(device)
    _, c = sae(x)
    active = (c > 0).float().sum(dim=-1)
    return active.mean().item()

avg_active = active_features_per_token(sae, tiny_subset)
print(f"avg features active per token: {avg_active:.1f} / {dict_size} total drawers")
print(f"that's {100 * avg_active / dict_size:.1f}% of all drawers open on average")

avg features active per token: 528.2 / 2048 total drawers
that's 25.8% of all drawers open on average


## 5c. Alpha sweep — find a sparsity strength that actually bites
Cheap to run on the tiny subset. Watch for: `sparsity` should keep decreasing (not plateau) across epochs, and the active-features-per-token count should drop meaningfully, while `recon` stays reasonably low. Expect `recon` to get a bit worse as `alpha` goes up -- that tradeoff is expected.

In [11]:
alpha_candidates = [1e-3, 1e-2, 5e-2, 1e-1]
sweep_results = {}

for a in alpha_candidates:
    print(f"\n=== alpha = {a} ===")
    sae_trial = SparseAutoencoder(d_model, dict_size).to(device)
    hist = train_sae(sae_trial, tiny_subset, a, epochs=30, log_every=29)
    avg_active = active_features_per_token(sae_trial, tiny_subset)
    final_epoch, final_loss, final_recon, final_sparsity = hist[-1]
    print(f"avg active features per token: {avg_active:.1f} / {dict_size} ({100*avg_active/dict_size:.1f}%)")
    sweep_results[a] = {
        "sae": sae_trial,
        "final_recon": final_recon,
        "final_sparsity": final_sparsity,
        "avg_active": avg_active,
    }

print("\n\nsummary:")
print(f"{'alpha':>8} | {'final recon':>12} | {'final sparsity':>15} | {'avg active/token':>17}")
for a, r in sweep_results.items():
    print(f"{a:>8} | {r['final_recon']:>12.2f} | {r['final_sparsity']:>15.2f} | {r['avg_active']:>17.1f}")


=== alpha = 0.001 ===
epoch   0 | loss 5132.189 | recon 5128.598 | sparsity 3591.077
epoch  29 | loss   63.185 | recon   61.114 | sparsity 2070.845
avg active features per token: 534.4 / 2048 (26.1%)

=== alpha = 0.01 ===
epoch   0 | loss 7632.248 | recon 7595.145 | sparsity 3710.323
epoch  29 | loss   89.282 | recon   68.795 | sparsity 2048.695
avg active features per token: 529.4 / 2048 (25.9%)

=== alpha = 0.05 ===
epoch   0 | loss 8269.608 | recon 8086.260 | sparsity 3666.969
epoch  29 | loss  172.179 | recon   77.008 | sparsity 1903.419
avg active features per token: 483.9 / 2048 (23.6%)

=== alpha = 0.1 ===
epoch   0 | loss 6080.080 | recon 5722.477 | sparsity 3576.031
epoch  29 | loss  244.463 | recon   73.510 | sparsity 1709.529
avg active features per token: 424.0 / 2048 (20.7%)


summary:
   alpha |  final recon |  final sparsity |  avg active/token
   0.001 |        61.11 |         2070.84 |             534.4
    0.01 |        68.79 |         2048.69 |             529.4
   

## 5d. Pick the best alpha and re-check interpretability
Set `chosen_alpha` below to whichever value from the summary table above gave a low avg-active-per-token count (roughly 10-50) with still-reasonable reconstruction, then re-run the top-activating-texts check on that model.

In [12]:
chosen_alpha = 1e-2  # <-- edit this after looking at the sweep summary table above

sae = sweep_results[chosen_alpha]["sae"]
print(f"using alpha = {chosen_alpha}")

for feature_idx in [0, 1, 2]:
    print(f"\n--- feature {feature_idx} ---")
    for act, text in top_activating_texts(sae, texts, feature_idx):
        print(f"{act:6.3f} | {text[:120]}")

using alpha = 0.01

--- feature 0 ---
 0.983 |  Partly due to these events , and partly due to the major losses in manpower Gallia suffers towards the end of the war w
 0.977 |  Lettering , sword , and shield ; mount for a list of men and woman serving in the Forces , St. Andrews , Croydon , 1943
 0.869 |  Of this number , approximately 9600 weapons were serviceable , or ready @-@ for @-@ issue . Note there were only 1 @,@ 
 0.838 |  The Confederate ordnance establishment at Little Rock was reactivated in August , 1862 . Looking around for a suitable 
 0.838 |  As with previous Valkyira Chronicles games , Valkyria Chronicles III is a tactical role @-@ playing game where players 

--- feature 1 ---
 1.248 |  The arsenal was constructed at the request of Governor James Sevier Conway in response to the perceived dangers of fron
 1.163 |  Barker 's sketches , drawings , and paintings of children were given to friends or to the parents of the subjects , don
 1.047 |  In the late 1920s , Bar

## 6. Logging (optional but recommended)
Swap the `print` calls above for Weights & Biases once the tiny run above shows the loss decreasing.

In [13]:
# !pip install -q wandb
# import wandb
# wandb.init(project="sae-reproduction", config={"alpha": alpha, "R": R, "layer": LAYER})
# for epoch, loss, recon, sparsity in history:
#     wandb.log({"epoch": epoch, "loss": loss, "recon": recon, "sparsity": sparsity})

## 7. Scale up (run this once everything above works)
Increase `N_DOCS` in section 2, re-run activation collection, then train longer here. Try sweeping `alpha` and `R`.

In [14]:
# N_DOCS = 5000  # go back to section 2, change this, and re-run collection first
# full_sae = SparseAutoencoder(d_model, dict_size).to(device)
# full_history = train_sae(full_sae, activations, alpha, epochs=200, log_every=10)

## 8. Quick manual interpretability check
For a given learned feature, find which real texts activate it most. This is a cheap first check before attempting the full GPT-4 autointerpretability pipeline.

In [15]:
# uses top_activating_texts, defined earlier in section 3b
for feature_idx in [0, 1, 2]:
    print(f"\n--- feature {feature_idx} ---")
    for act, text in top_activating_texts(sae, texts, feature_idx):
        print(f"{act:6.3f} | {text[:120]}")


--- feature 0 ---
 0.983 |  Partly due to these events , and partly due to the major losses in manpower Gallia suffers towards the end of the war w
 0.977 |  Lettering , sword , and shield ; mount for a list of men and woman serving in the Forces , St. Andrews , Croydon , 1943
 0.869 |  Of this number , approximately 9600 weapons were serviceable , or ready @-@ for @-@ issue . Note there were only 1 @,@ 
 0.838 |  The Confederate ordnance establishment at Little Rock was reactivated in August , 1862 . Looking around for a suitable 
 0.838 |  As with previous Valkyira Chronicles games , Valkyria Chronicles III is a tactical role @-@ playing game where players 

--- feature 1 ---
 1.248 |  The arsenal was constructed at the request of Governor James Sevier Conway in response to the perceived dangers of fron
 1.163 |  Barker 's sketches , drawings , and paintings of children were given to friends or to the parents of the subjects , don
 1.047 |  In the late 1920s , Barker began to doubt 